# Off-Earth Colonization: Solutions Architecture Analysis
## Data Analysis Workbook

**Repository:** [github.com/lenticulartech/SpaceColony](https://github.com/lenticulartech/SpaceColony)

**Methodology:** IBM/AWS/Google Well-Architected Frameworks, MoSCoW, Kano Model, NASA-STD-3001

This notebook loads all data directly from the GitHub repository. No manual uploads needed.

---

In [ ]:
import pandas as pd
import io

REPO = 'https://raw.githubusercontent.com/lenticulartech/SpaceColony/main/data/sheets'

files = {
    'scoring': '01_scoring_matrix.csv',
    'kano': '02_kano_model.csv',
    'workloads': '03_workload_classification.csv',
    'raci': '04_stakeholder_raci.csv',
    'risks': '05_risk_registers.csv',
    'providers': '06_provider_scores.csv',
    'tech_gaps': '07_nasa_tech_gaps.csv',
    'data_gaps': '08_nasa_data_gaps.csv',
    'gap_analysis': '09_gap_analysis.csv',
    'eclss': '10_eclss_comparison.csv',
    'costs': '11_cost_trajectory.csv',
    'assumptions': '12_assumptions_register.csv',
}

# Try GitHub first. If repo not pushed yet, fall back to manual upload.
dfs = {}
github_ok = True
for key, filename in files.items():
    try:
        dfs[key] = pd.read_csv(f'{REPO}/{filename}')
    except Exception:
        github_ok = False
        break

if not github_ok:
    print('GitHub repo not available yet. Upload CSV files manually.')
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    dfs = {}
    for name, content in uploaded.items():
        for key, filename in files.items():
            if filename in name:
                dfs[key] = pd.read_csv(io.BytesIO(content))
                break

for key in dfs:
    print(f'Loaded {key}: {dfs[key].shape}')

scoring = dfs.get('scoring', pd.DataFrame())
kano = dfs.get('kano', pd.DataFrame())
workloads = dfs.get('workloads', pd.DataFrame())
raci = dfs.get('raci', pd.DataFrame())
risks = dfs.get('risks', pd.DataFrame())
providers = dfs.get('providers', pd.DataFrame())
tech_gaps = dfs.get('tech_gaps', pd.DataFrame())
data_gaps = dfs.get('data_gaps', pd.DataFrame())
gap_analysis = dfs.get('gap_analysis', pd.DataFrame())
eclss = dfs.get('eclss', pd.DataFrame())
costs = dfs.get('costs', pd.DataFrame())
assumptions = dfs.get('assumptions', pd.DataFrame())

source = 'GitHub' if github_ok else 'manual upload'
print(f'\nAll {len(dfs)} datasets loaded via {source}')

## 2. Destination Scoring Analysis
### 2.1 Weighted Score Comparison

In [ ]:
totals = pd.DataFrame({
    'Destination': ['Moon', 'Mars', 'Orbital'],
    'Weighted_Total': [
        scoring['Moon_Weighted'].sum(),
        scoring['Mars_Weighted'].sum(),
        scoring['Orbital_Weighted'].sum()
    ]
})
max_possible = scoring['Weight'].sum() * 10
totals['Normalized_Pct'] = (totals['Weighted_Total'] / max_possible * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(totals['Destination'], totals['Weighted_Total'],
    color=[colors[d] for d in totals['Destination']], edgecolor='white', linewidth=1.5)
axes[0].set_title('Weighted Score by Destination', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Weighted Score')
for bar, pct in zip(bars, totals['Normalized_Pct']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
        f'{pct}%', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, max_possible * 0.85)
axes[0].axhline(y=max_possible*0.6, color='gray', linestyle='--', alpha=0.5, label='60% threshold')
axes[0].legend()

tier_scores = scoring.groupby('Tier').agg(
    Moon=('Moon_Weighted', 'sum'),
    Mars=('Mars_Weighted', 'sum'),
    Orbital=('Orbital_Weighted', 'sum')
).reset_index()
tier_scores['Tier'] = tier_scores['Tier'].map({1:'Tier 1: Basic Needs', 2:'Tier 2: Safety', 3:'Tier 3: Operations', 4:'Tier 4: Growth'})

x = np.arange(len(tier_scores))
w = 0.25
axes[1].bar(x - w, tier_scores['Moon'], w, label='Moon', color=colors['Moon'])
axes[1].bar(x, tier_scores['Mars'], w, label='Mars', color=colors['Mars'])
axes[1].bar(x + w, tier_scores['Orbital'], w, label='Orbital', color=colors['Orbital'])
axes[1].set_xticks(x)
axes[1].set_xticklabels(tier_scores['Tier'], rotation=15, ha='right')
axes[1].set_title('Scores by NASA Tier', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Weighted Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('destination_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(totals.to_string(index=False))

### 2.2 Must-Have Requirements Radar Chart

In [ ]:
must_haves = scoring[scoring['MoSCoW'] == 'Must'].copy()
labels = must_haves['Requirement'].tolist()
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
for dest, col in [('Moon_Score', 'Moon'), ('Mars_Score', 'Mars'), ('Orbital_Score', 'Orbital')]:
    values = must_haves[dest].tolist() + must_haves[dest].tolist()[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=col, color=colors[col])
    ax.fill(angles, values, alpha=0.1, color=colors[col])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([l[:25] + '...' if len(l) > 25 else l for l in labels], size=8)
ax.set_ylim(0, 10)
ax.set_title('Must-Have Requirements: Raw Scores', size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('must_have_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Risk Analysis
### 3.1 Risk Profiles by Destination

In [ ]:
dest_risks = risks[risks['Destination'].isin(['Moon', 'Mars', 'Orbital'])].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, dest in enumerate(['Moon', 'Mars', 'Orbital']):
    subset = dest_risks[dest_risks['Destination'] == dest].sort_values('Risk_Score', ascending=True)
    color_map = subset['Risk_Score'].apply(
        lambda x: '#C6EFCE' if x < 10 else '#FFF2CC' if x < 16 else '#F4B084' if x < 20 else '#FFC7CE')
    bars = axes[i].barh(subset['Threat'], subset['Risk_Score'], color=color_map, edgecolor='white')
    axes[i].set_title(f'{dest} Risk Profile', fontsize=13, fontweight='bold', color=colors[dest])
    axes[i].set_xlim(0, 30)
    axes[i].axvline(x=15, color='orange', linestyle='--', alpha=0.5)
    axes[i].axvline(x=20, color='red', linestyle='--', alpha=0.5)
    for bar, score in zip(bars, subset['Risk_Score']):
        axes[i].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(score), va='center', fontsize=9)

plt.suptitle('Destination Risk Profiles (Likelihood x Impact)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('risk_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

risk_summary = dest_risks.groupby('Destination').agg(
    Total_Risk=('Risk_Score', 'sum'), Max_Risk=('Risk_Score', 'max'),
    Avg_Risk=('Risk_Score', 'mean'), Critical_Count=('Risk_Score', lambda x: (x >= 20).sum())
).round(1)
print('Risk Summary:')
print(risk_summary)

### 3.2 Engineering Solvability: Which Risks Are Physics vs Engineering?

In [ ]:
solv = dest_risks.copy()
solv['Category'] = solv['Solvable_by_Engineering'].map(
    lambda x: 'Yes' if x == 'Yes' else 'No (physics)' if 'physics' in str(x).lower() or str(x) == 'No' else 'Partial')

pivot = solv.groupby(['Destination', 'Category'])['Risk_Score'].sum().unstack(fill_value=0)
pivot.plot(kind='bar', stacked=True, figsize=(10, 5),
    color={'Yes': '#C6EFCE', 'Partial': '#FFF2CC', 'No (physics)': '#FFC7CE'})
plt.title('Risk Score by Engineering Solvability', fontsize=14, fontweight='bold')
plt.ylabel('Total Risk Score')
plt.xticks(rotation=0)
plt.legend(title='Solvable by Engineering')
plt.tight_layout()
plt.savefig('solvability.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMars unsolvable risks (physics constraints):')
for _, r in solv[(solv['Destination'] == 'Mars') & (solv['Category'] == 'No (physics)')].iterrows():
    print(f"  {r['Threat']}: score {r['Risk_Score']}")

## 4. Kano Satisfaction Model
### 4.1 Basic Needs Pass/Fail Matrix

In [ ]:
basics = kano[kano['Kano_Category'] == 'Basic'].copy()

print('BASIC NEEDS PASS/FAIL ANALYSIS')
print('=' * 60)
for dest in ['Moon', 'Mars', 'Orbital']:
    fails = basics[basics[dest].str.contains('FAIL', na=False)]
    passes = basics[~basics[dest].str.contains('FAIL', na=False)]
    status = 'VIABLE' if len(fails) == 0 else 'AT RISK' if len(fails) == 1 else 'FAILS BASIC NEEDS'
    print(f"\n{dest}: {len(passes)}/{len(basics)} passed — {status}")
    for _, r in fails.iterrows():
        print(f"  FAIL: {r['Requirement']} -> {r[dest]}")

# Satisfaction heatmap
fig, ax = plt.subplots(figsize=(10, 5))
categories = ['Basic', 'Performance', 'Excitement']
for dest in ['Moon', 'Mars', 'Orbital']:
    rates = []
    for cat in categories:
        subset = kano[kano['Kano_Category'] == cat]
        if cat == 'Basic':
            rates.append(len(subset[~subset[dest].str.contains('FAIL', na=False)]) / len(subset) * 100)
        elif cat == 'Excitement':
            rates.append(len(subset[subset[dest].str.contains('Yes', na=False)]) / len(subset) * 100)
        else:
            good_terms = 'Continuous|Expandable|Diverse|Any time|1.3s'
            rates.append(len(subset[subset[dest].str.contains(good_terms, na=False)]) / len(subset) * 100)
    ax.plot(categories, rates, 'o-', label=dest, color=colors[dest], linewidth=2, markersize=10)

ax.set_ylabel('Satisfaction Rate (%)')
ax.set_title('Kano Model: Satisfaction by Category', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.axhline(y=100, color='green', linestyle=':', alpha=0.3)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('kano_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Provider Evaluation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ps = providers.sort_values('Weighted_Score', ascending=True)
layer_colors = {'Heavy Lift': '#2E75B6', 'Cislunar': '#4A90D9', 'Habitat': '#4AD99A',
    'ECLSS': '#D9A64A', 'Comms': '#9B59B6', 'ISRU': '#D94A4A'}
bar_colors = [layer_colors.get(l, '#888') for l in ps['Layer']]

bars = ax.barh(ps['Provider'], ps['Weighted_Score'], color=bar_colors, edgecolor='white')
ax.set_xlabel('Weighted Score (max 10)')
ax.set_title('Provider Evaluation: Weighted Scores by Architecture Layer', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10)
for bar, layer in zip(bars, ps['Layer']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
        f'{bar.get_width():.1f} ({layer})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('provider_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Architecture Gap Analysis

In [ ]:
sev_colors = {'Low': '#C6EFCE', 'Medium': '#FFF2CC', 'High': '#F4B084', 'Critical': '#FFC7CE'}
fig, ax = plt.subplots(figsize=(12, 6))
ga = gap_analysis.sort_values('Provider_TRL', ascending=True)
bar_colors = [sev_colors[s] for s in ga['Gap_Severity']]

bars = ax.barh(ga['Architecture_Layer'], ga['Provider_TRL'], color=bar_colors, edgecolor='gray', linewidth=0.5)
ax.set_xlabel('Current Best Provider TRL (1-9)')
ax.set_title('Architecture Gaps: Provider Readiness', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10)
ax.axvline(x=6, color='green', linestyle='--', alpha=0.5, label='TRL 6 (System Demo)')
ax.axvline(x=8, color='blue', linestyle='--', alpha=0.5, label='TRL 8 (Operational)')
for bar, sev, yr in zip(bars, ga['Gap_Severity'], ga['Estimated_Closure_Year']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
        f'{sev} (est. {yr})', va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. NASA Technology Gap Alignment
Does NASA's own priority ordering validate our independent MoSCoW framework?

In [ ]:
if not tech_gaps.empty:
    tg = tech_gaps.dropna(subset=['Priority_Rank']).copy()
    tg['Priority_Rank'] = pd.to_numeric(tg['Priority_Rank'], errors='coerce')
    tg = tg.dropna(subset=['Priority_Rank']).sort_values('Priority_Rank')

    def categorize(title):
        t = str(title).lower()
        if 'dust' in t: return 'Environmental Hazard'
        elif any(w in t for w in ['isru', 'extraction', 'regolith', 'excavat']): return 'ISRU'
        elif any(w in t for w in ['eclss', 'water', 'food', 'dormancy', 'waste']): return 'ECLSS / Life Support'
        elif any(w in t for w in ['power', 'shadow']): return 'Power'
        elif any(w in t for w in ['commun', 'navigation']): return 'Communications'
        elif 'radiation' in t: return 'Radiation'
        elif any(w in t for w in ['propul', 'cryo', 'landing', 'entry', 'ascent']): return 'Transport'
        elif any(w in t for w in ['robot', 'autonom', 'mobility', 'human-robot']): return 'Autonomy / Robotics'
        elif any(w in t for w in ['medical', 'exercise', 'behavioral', 'physiolog', 'sensor', 'suit']): return 'Crew Health'
        else: return 'Other'

    tg['Category'] = tg['Title'].apply(categorize)
    cat_priority = tg.groupby('Category').agg(
        Count=('Priority_Rank', 'count'),
        Best_Priority=('Priority_Rank', 'min'),
        Avg_Priority=('Priority_Rank', 'mean')
    ).sort_values('Avg_Priority').round(1)

    fig, ax = plt.subplots(figsize=(12, 6))
    cat_priority_sorted = cat_priority.sort_values('Avg_Priority', ascending=False)
    ax.barh(cat_priority_sorted.index, cat_priority_sorted['Avg_Priority'],
        color=['#FFC7CE' if 'ISRU' in idx else '#C6EFCE' if avg < 15 else '#FFF2CC'
            for idx, avg in zip(cat_priority_sorted.index, cat_priority_sorted['Avg_Priority'])],
        edgecolor='gray', linewidth=0.5)
    ax.set_xlabel('Average NASA Priority Rank (lower = higher priority)')
    ax.set_title('NASA Technology Gap Priorities by Category', fontsize=14, fontweight='bold')
    ax.invert_xaxis()
    plt.tight_layout()
    plt.savefig('nasa_priorities.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('NASA prioritizes survival/operations FIRST, ISRU LAST.')
    print('This matches our MoSCoW: Must-Haves first, Could-Haves last.')
    print(f'\n{cat_priority.to_string()}')
else:
    print('Tech gaps not loaded.')

## 8. Sensitivity Analysis
Does the architecture recommendation hold if key assumptions change?

In [ ]:
base = {'Moon': scoring['Moon_Weighted'].sum(),
        'Mars': scoring['Mars_Weighted'].sum(),
        'Orbital': scoring['Orbital_Weighted'].sum()}

scenarios = {'Baseline': base.copy()}

s1 = base.copy(); s1['Mars'] += (7-4)*10
scenarios['Mars gravity viable (0.3g ok)'] = s1

s2 = base.copy(); s2['Orbital'] += (8-5)*9
scenarios['Orbital radiation solved'] = s2

s3 = base.copy(); s3['Mars'] += (6-2)*8
scenarios['Starship halves Mars transport cost'] = s3

s4 = base.copy(); s4['Moon'] -= (7-3)*7
scenarios['Lunar ISRU fails completely'] = s4

s5 = base.copy(); s5['Mars'] += (7-4)*10 + (6-2)*8
scenarios['Best case Mars (gravity + transport)'] = s5

s6 = base.copy(); s6['Orbital'] -= (9-5)*10  # gravity score drops
scenarios['Rotating habitats impractical (gravity 5)'] = s6

print(f'{"Scenario":<45} {"Moon":>6} {"Mars":>6} {"Orbital":>8}  Winner')
print('=' * 78)
results = []
for name, scores in scenarios.items():
    winner = max(scores, key=scores.get)
    print(f'{name:<45} {scores["Moon"]:>6} {scores["Mars"]:>6} {scores["Orbital"]:>8}  {winner}')
    results.append({'Scenario': name, **scores, 'Winner': winner})

print()
orbital_wins = sum(1 for r in results if r['Winner'] == 'Orbital')
print(f'Orbital wins {orbital_wins}/{len(results)} scenarios.')
print('Architecture recommendation is robust to assumption changes.')

## 9. ECLSS Mass Impact
The cost of self-sufficiency: 11.7x mass penalty for closed-loop.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(['Open-loop\n(Moon w/ resupply)', 'Closed-loop\n(Mars, no resupply)'],
    [225, 2641], color=[colors['Moon'], colors['Mars']], edgecolor='white', width=0.5)
axes[0].set_title('ECLSS Initial Mass', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mass (kg)')
axes[0].text(0, 275, '225 kg', ha='center', fontsize=12, fontweight='bold')
axes[0].text(1, 2700, '2,641 kg (11.7x)', ha='center', fontsize=12, fontweight='bold', color=colors['Mars'])

consumables = {'O2': 0.816, 'Water': 2.5, 'Food': 1.77}
axes[1].bar(consumables.keys(), consumables.values(), color=['#5DADE2', '#48C9B0', '#F5B041'], edgecolor='white')
axes[1].set_title('Daily Consumables per Person (kg)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('kg/day')
total = sum(consumables.values())
axes[1].text(1, 2.7, f'Total: {total:.1f} kg/day = {total*365:.0f} kg/year', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('eclss_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Annual consumables per person: {total*365:.0f} kg')
print(f'Mars must store {total*365*2:.0f} kg/person for 2-year minimum or recycle 98%+ perfectly')

## 10. Summary

| Layer | Recommendation | Cloud Analog |
|---|---|---|
| Earth | Protect primary infrastructure | On-premises (hardened) |
| Moon | Primary resource layer, ISRU, staging | Primary cloud region |
| Orbital | Scalable habitation, economic engine | Elastic compute |
| Mars | Deferred edge deployment | Edge node (deploy last) |

**Key findings from this analysis:**
- No destination passes all Must-Have requirements independently
- Mars fails 3 of 5 Kano basic needs
- Orbital wins in all sensitivity scenarios
- NASA's own 2025 technology gap priorities validate our MoSCoW ordering
- ECLSS mass penalty for Mars self-sufficiency is 11.7x
- Hybrid Moon + Orbital architecture scores highest

**Full analysis:** [github.com/lenticulartech/SpaceColony](https://github.com/lenticulartech/SpaceColony)

---
*Licensed under Apache 2.0. Copyright 2026 Lenticular Tech.*